# Data Cleaning and Preprocessing

## Objective

This notebook focuses on preparing the Global Weather Repository dataset for exploratory data analysis, advanced analysis, and forecasting.

The main preprocessing steps include:
- Loading and inspecting the dataset
- Checking missing values and duplicate records
- Converting timestamp columns into datetime format
- Extracting useful time-based features
- Detecting potential outliers
- Creating additional weather-related features
- Saving the cleaned dataset for later analysis

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Dataset Overview

The dataset contains global weather observations collected from cities around the world. It includes meteorological, atmospheric, air quality, and astronomical features.

The dataset provides:
- Temperature and humidity measurements
- Wind and pressure information
- Air quality indicators
- Visibility and precipitation data
- Geographical information
- Sunrise/sunset and lunar attributes
- Timestamped weather observations

This rich feature set enables both forecasting and environmental trend analysis.

In [2]:
DATA_PATH = Path("../data/raw/GlobalWeatherRepository.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (139948, 41)


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,wind_mph,wind_kph,wind_degree,wind_direction,pressure_mb,pressure_in,precip_mm,precip_in,humidity,cloud,feels_like_celsius,feels_like_fahrenheit,visibility_km,visibility_miles,uv_index,gust_mph,gust_kph,air_quality_Carbon_Monoxide,air_quality_Ozone,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,8.3,13.3,338,NNW,1012.0,29.89,0.0,0.00,24,30,25.3,77.5,10.0,6.0,7.0,9.5,15.3,277.0,103.0,1.1,0.2,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,6.9,11.2,320,NW,1012.0,29.88,0.1,0.00,94,75,19.0,66.2,10.0,6.0,5.0,11.4,18.4,193.6,97.3,0.9,0.1,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,9.4,15.1,280,W,1011.0,29.85,0.0,0.00,29,0,24.6,76.4,10.0,6.0,5.0,13.9,22.3,540.7,12.2,65.1,13.4,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,7.4,11.9,215,SW,1007.0,29.75,0.3,0.01,61,100,3.8,38.9,2.0,1.0,2.0,8.5,13.7,170.2,64.4,1.6,0.2,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,8.1,13.0,150,SSE,1011.0,29.85,0.0,0.00,89,50,28.7,83.6,10.0,6.0,8.0,12.5,20.2,2964.0,19.0,72.7,31.5,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


# Initial Data Inspection

The first step is to understand the structure and quality of the dataset.

This includes:
- Checking dataset dimensions
- Inspecting column datatypes
- Reviewing summary statistics
- Identifying missing values and duplicates

In [3]:
df.info()
df.describe(include="all").T

<class 'pandas.DataFrame'>
RangeIndex: 139948 entries, 0 to 139947
Data columns (total 41 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   country                       139948 non-null  str    
 1   location_name                 139948 non-null  str    
 2   latitude                      139948 non-null  float64
 3   longitude                     139948 non-null  float64
 4   timezone                      139948 non-null  str    
 5   last_updated_epoch            139948 non-null  int64  
 6   last_updated                  139948 non-null  str    
 7   temperature_celsius           139948 non-null  float64
 8   temperature_fahrenheit        139948 non-null  float64
 9   condition_text                139948 non-null  str    
 10  wind_mph                      139948 non-null  float64
 11  wind_kph                      139948 non-null  float64
 12  wind_degree                   139948 non-null  int64  


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
country,139948,211,Bulgaria,1656,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_name,139948,257,Kabul,720,NaN,NaN,NaN,NaN,NaN,NaN,NaN
latitude,139948.0,NaN,NaN,NaN,19.21493,24.413648,-41.3,4.0503,17.25,40.4,64.15
longitude,139948.0,NaN,NaN,NaN,21.944697,65.784957,-175.2,-6.8361,23.2361,49.8822,179.22
timezone,139948,199,Asia/Bangkok,2568,NaN,NaN,NaN,NaN,NaN,NaN,NaN
last_updated_epoch,139948.0,NaN,NaN,NaN,1746994231.944722,17976485.877554,1715849100.0,1731488400.0,1747040400.0,1762503300.0,1778135400.0
last_updated,139948,23224,2025-12-26 08:15,45,NaN,NaN,NaN,NaN,NaN,NaN,NaN
temperature_celsius,139948.0,NaN,NaN,NaN,21.244525,9.660899,-29.8,15.6,23.8,28.0,79.3
temperature_fahrenheit,139948.0,NaN,NaN,NaN,70.241942,17.389484,-21.6,60.1,74.8,82.4,174.7
condition_text,139948,49,Sunny,40281,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df)) * 100
}).sort_values(by="missing_percentage", ascending=False)

missing_summary.head(20)

,missing_count,missing_percentage
country,0,0.0
location_name,0,0.0
latitude,0,0.0
longitude,0,0.0
timezone,0,0.0
last_updated_epoch,0,0.0
last_updated,0,0.0
temperature_celsius,0,0.0
temperature_fahrenheit,0,0.0
condition_text,0,0.0


In [5]:
duplicate_count = df.duplicated().sum()
print("Duplicate count:", duplicate_count)

Duplicate count: 0


In [6]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.columns.tolist()

['country',
 'location_name',
 'latitude',
 'longitude',
 'timezone',
 'last_updated_epoch',
 'last_updated',
 'temperature_celsius',
 'temperature_fahrenheit',
 'condition_text',
 'wind_mph',
 'wind_kph',
 'wind_degree',
 'wind_direction',
 'pressure_mb',
 'pressure_in',
 'precip_mm',
 'precip_in',
 'humidity',
 'cloud',
 'feels_like_celsius',
 'feels_like_fahrenheit',
 'visibility_km',
 'visibility_miles',
 'uv_index',
 'gust_mph',
 'gust_kph',
 'air_quality_carbon_monoxide',
 'air_quality_ozone',
 'air_quality_nitrogen_dioxide',
 'air_quality_sulphur_dioxide',
 'air_quality_pm2.5',
 'air_quality_pm10',
 'air_quality_us_epa_index',
 'air_quality_gb_defra_index',
 'sunrise',
 'sunset',
 'moonrise',
 'moonset',
 'moon_phase',
 'moon_illumination']

# Datetime Parsing and Time Feature Extraction

The `last_updated` column was converted into datetime format to enable time series analysis and temporal feature engineering.

Additional time-based features were extracted, including:
- Year
- Month
- Day
- Hour
- Day of year
- Weekday

These features help identify seasonal and temporal weather patterns.

In [9]:
df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")

print("Invalid dates:", df["last_updated"].isnull().sum())
print("Date range:", df["last_updated"].min(), "to", df["last_updated"].max())

Invalid dates: 0
Date range: 2024-05-16 01:45:00 to 2026-05-07 19:30:00


# Feature Engineering

Additional derived features were created to improve exploratory analysis and forecasting capability.

The engineered features include:
- `is_weekend` → Indicates whether the observation occurred on a weekend
- `is_day` → Distinguishes daytime from nighttime observations
- `season` → Categorizes observations into seasonal groups
- `temp_difference` → Difference between actual and perceived temperature

These features help capture environmental and temporal behavior more effectively.

In [11]:
# Time-based features
df["year"] = df["last_updated"].dt.year
df["month"] = df["last_updated"].dt.month
df["day"] = df["last_updated"].dt.day
df["hour"] = df["last_updated"].dt.hour
df["day_of_year"] = df["last_updated"].dt.dayofyear
df["weekday"] = df["last_updated"].dt.day_name()

# Weekend indicator
df["is_weekend"] = df["weekday"].isin(
    ["Saturday", "Sunday"]
).astype(int)

# Day/Night indicator
df["is_day"] = df["hour"].apply(
    lambda x: 1 if 6 <= x <= 18 else 0
)

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

df["season"] = df["month"].apply(get_season)

df["temp_difference"] = (
    df["feels_like_celsius"] - df["temperature_celsius"]
)

# Outlier Detection and Analysis

Outlier analysis was performed using the Interquartile Range (IQR) method to identify unusually extreme observations across numerical weather variables.

Several variables showed a high proportion of outliers, particularly:
- Visibility
- Precipitation
- Air quality indicators

However, these values were retained because weather datasets naturally contain extreme environmental events such as:
- Storms
- Heavy rainfall
- Fog and low visibility
- Pollution spikes
- Heatwaves

Removing these observations could eliminate meaningful climatic behavior and reduce forecasting realism.

Therefore, the detected outliers were treated as potential environmental anomalies rather than invalid records.

In [12]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

outlier_summary = []

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    
    outlier_summary.append({
        "feature": col,
        "outlier_count": outliers,
        "outlier_percentage": (outliers / len(df)) * 100
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values(
    by="outlier_percentage", ascending=False
)

outlier_df.head(20)

,feature,outlier_count,outlier_percentage
16,visibility_km,29909,21.371509
17,visibility_miles,29740,21.250750
31,is_day,28584,20.424729
10,precip_mm,28397,20.291108
11,precip_in,21532,15.385715
24,air_quality_sulphur_dioxide,19274,13.772258
26,air_quality_pm10,14932,10.669677
23,air_quality_nitrogen_dioxide,14486,10.350988
28,air_quality_gb_defra_index,13093,9.355618
21,air_quality_carbon_monoxide,12527,8.951182


# Final Processed Dataset

After preprocessing and feature engineering, the dataset was saved for downstream exploratory analysis and forecasting tasks.

The processed dataset now contains:
- Cleaned datetime features
- Temporal indicators
- Seasonal features
- Environmental and weather attributes
- Air quality measurements

This processed dataset will be used for:
- Exploratory Data Analysis (EDA)
- Climate trend analysis
- Forecasting model development
- Spatial and geographical analysis

In [16]:
from pathlib import Path

processed_path = Path("../data/processed/weather_cleaned.csv")

df.to_csv(processed_path, index=False)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.
